In [ ]:
%cd ..

In [ ]:
from pathlib import Path
from mol_gen_docking.data.pydantic_dataset import read_jsonl
import json

In [24]:
molgendata= Path("data/molgendata/test_data/test_prompts_ood.jsonl")
sys_prompt = json.load(open("system_prompts/vanilla.json"))

data = read_jsonl(molgendata)

In [25]:
def sample_to_request(sample, max_tokens: int=32768, temperature: float=0.7, reasoning_effort: str = "high", n=64):
    id = sample.identifier
    messages = [
        sys_prompt,
        {
            "role": "user", "content": sample.conversations[0].messages[-1].content
        }
    ]
    return dict(
        custom_id = id,
        body = dict(
            max_tokens = max_tokens,
            temperature = temperature,
            reasoning_effort = reasoning_effort,
            n=n,
            messages=messages
        )
    )


requests = [sample_to_request(sample) for sample in data]

In [26]:
requests

[{'custom_id': '334810',
  'body': {'max_tokens': 32768,
   'temperature': 0.7,
   'reasoning_effort': 'high',
   'n': 64,
   'messages': [{'role': 'system',
     'content': 'You are a molecular modeling assistant. You answer to questions asked by a user in english.\\nYou can first draft your thinking process or inner monologue before arriving to an answer.\\nYou must provide your final answer enclosed in <answer> </answer> tags e.g <answer> answer here </answer>.'},
    {'role': 'user',
     'content': 'I am working on designing a drug-like compound meeting a specific objective to be a possible drug. Given the objective to optimize, propose a drug-like molecule as a SMILES string. Here is the objective to optimize: \n    - Minimize Docking score against spodoptera ecdysone receptor.\n(The docking score represents the free-energy of binding, low scores corresponding to strong binders.)'}]}},
 {'custom_id': '30810',
  'body': {'max_tokens': 32768,
   'temperature': 0.7,
   'reasoning_ef

In [27]:
# save as a jsonl file
with open("notebooks/molgendata_ms_batch_api.jsonl", "w") as f:
    for request in requests:
        f.write(json.dumps(request) + "\n")

In [28]:
from mistralai.client import Mistral
import os

In [29]:
api_key = os.environ["MISTRAL_API_KEY"]
client = Mistral(api_key=api_key)

for m in client.models.list().data:
    if m.name and "small" in m.name.lower():
        print(f"ID: {m.id} | Name: {m.name} | aliases: {';'.join(m.aliases)}")

ID: mistral-small-2603 | Name: mistral-small-2603 | aliases: mistral-small-latest;mistral-vibe-cli-fast
ID: mistral-small-latest | Name: mistral-small-2603 | aliases: mistral-small-2603;mistral-vibe-cli-fast
ID: mistral-vibe-cli-fast | Name: mistral-small-2603 | aliases: mistral-small-2603;mistral-small-latest
ID: mistral-small-2506 | Name: mistral-small-2506 | aliases: 
ID: magistral-small-2509 | Name: magistral-small-2509 | aliases: magistral-small-latest
ID: magistral-small-latest | Name: magistral-small-2509 | aliases: magistral-small-2509
ID: voxtral-small-2507 | Name: voxtral-small-2507 | aliases: voxtral-small-latest
ID: voxtral-small-latest | Name: voxtral-small-2507 | aliases: voxtral-small-2507
ID: devstral-small-2507 | Name: devstral-small-2507 | aliases: 
ID: labs-mistral-small-creative | Name: labs-mistral-small-creative | aliases: 
ID: openai/text-embedding-3-small | Name: openai/text-embedding-3-small | aliases: 


In [30]:
batch_data = client.files.upload(
    file={
        "file_name": "test.jsonl",
        "content": open("notebooks/molgendata_ms_batch_api.jsonl", "rb")
    },
    purpose = "batch"
)

In [31]:
created_job = client.batch.jobs.create(
    input_files=[batch_data.id],
    model="mistral-small-latest",
    endpoint="/v1/chat/completions",
    metadata={"job_type": "testing"}
)

In [21]:
retrieved_job = client.batch.jobs.get(job_id=created_job.id)

if retrieved_job.status == "QUEUED":
    print("Waiting for job to complete...")
else:
    try:
        output_file_stream = client.files.download(file_id=retrieved_job.error_file)
        error = json.load(output_file_stream)["response"]["body"]
        print(json.loads(error)["message"])
    except Exception as e:
        print("No errors")
        output_file_stream = client.files.download(file_id=retrieved_job.output_file)

No errors


In [22]:
retrieved_job

BatchJob(id='0781a0f9-8024-4ef0-9303-48a7099b34f1', input_files=['d06fef34-a6bf-436d-9f7c-bc9759c44f53'], endpoint='/v1/chat/completions', errors=[], status='SUCCESS', created_at=1776428930, total_requests=1, completed_requests=1, succeeded_requests=1, failed_requests=0, object='batch', metadata={'job_type': 'testing'}, model='mistral-small-latest', agent_id=None, output_file='01dca3d5-21bb-4f07-8e0a-db043cdd2911', error_file=None, outputs=None, started_at=1776428931, completed_at=1776429225)

In [23]:
# Write and save the file
with open('notebooks/batch_results.jsonl', 'wb') as f:
    f.write(output_file_stream.read())